# Panel-data pilot 2: FLUX.2 klein 4B, previous-panel reference and stronger change wording

Modes: c = previous panel as reference, same prompts as pilot 1 mode b. d = previous panel as reference plus prompts that say what changes. e = panel 1 as reference plus the d prompts (isolates the wording effect). Same seeds as pilot 1. Needs GPU T4 x2 and Internet on.

In [ ]:
import subprocess, sys
print(subprocess.run("pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece huggingface_hub 2>&1 | tail -3", shell=True, capture_output=True, text=True).stdout)
import diffusers, transformers, torch
print("diffusers", diffusers.__version__, "transformers", transformers.__version__, "torch", torch.__version__)

In [ ]:
import subprocess, torch
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout)
N_GPU = torch.cuda.device_count()
GPU_INFO = []
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    GPU_INFO.append({"index": i, "name": p.name, "total_gb": round(p.total_memory / 1e9, 2), "capability": f"{p.major}.{p.minor}", "bf16_native": torch.cuda.is_bf16_supported(including_emulation=False) if i == 0 else None})
    print(GPU_INFO[-1])
import os
os.makedirs("/kaggle/working/out", exist_ok=True)
print("RAM:", subprocess.run("free -g | head -2", shell=True, capture_output=True, text=True).stdout)
print("disk:", subprocess.run("df -h /kaggle/working /root | tail -2", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Concept scripts (same content as scripts/concepts_pilot2.json)
CONCEPTS_JSON = r'''{
  "model": "black-forest-labs/FLUX.2-klein-4B",
  "style_prefix": "Flat vector illustration, simple shapes, limited palette, white background, no text, no letters.",
  "reference_instruction": "Keep exactly the same visual style, colour palette and objects as the reference image, only change the moment shown.",
  "change_instruction": "Edit the reference image: keep exactly the same objects, colours, style and position of everything that is not mentioned below. What changes:",
  "width": 512,
  "height": 512,
  "steps": 4,
  "note": "Pilot 2. Seeds identical to pilot 1. worst_under_acting: config, ratelimit, auth; worked: queue, ci, merge.",
  "concepts": [
    {
      "id": "config",
      "title": "Config loader with schema validation",
      "seed": 1002,
      "group": "under-acted",
      "panels": [
        "a small orange file (the config) slides in from the left toward a blue gate",
        "the orange file is placed against a blue stencil shaped like a template to compare its shape",
        "one corner of the orange file does not fit the blue stencil and a red mark appears on it",
        "the orange file has a corrected corner, fits the blue stencil exactly and a green glow appears"
      ],
      "change_panels": [
        "The orange file has moved right and now touches the blue gate; a thin blue outline of a file shape appears on the blue gate, the same size as the orange file.",
        "One corner of the orange file now sticks out past the blue outline, and a bright red circle appears on that corner.",
        "The corner of the orange file is now cut straight and fits inside the blue outline exactly; the red circle is gone and a green glow surrounds the orange file."
      ]
    },
    {
      "id": "ratelimit",
      "title": "Rate limiting",
      "seed": 1009,
      "group": "under-acted",
      "panels": [
        "a crowd of small orange dots (requests) rushes from the left toward a blue gate",
        "the blue gate lets a few orange dots through one at a time while the others bunch up behind it",
        "the other orange dots are held back by a grey bar across the gate, a small dial on the gate points high",
        "the grey bar lifts again and orange dots flow through steadily in a thin line to the server on the right"
      ],
      "change_panels": [
        "Most of the orange dots are now bunched in a tight cluster in front of the blue gate, and only three orange dots are passing through the blue gate.",
        "A thick grey bar now blocks the opening of the blue gate completely; all the orange dots are piled up against the grey bar on the left and none are on the right side.",
        "The thick grey bar has lifted up above the blue gate; a thin single line of orange dots now flows through the open blue gate to the right side."
      ]
    },
    {
      "id": "auth",
      "title": "Authentication tokens",
      "seed": 1007,
      "group": "under-acted",
      "panels": [
        "a small blue circle (the user) shows a small grey key to a blue door guard at a gate",
        "the blue door guard hands the blue circle a small round orange token in exchange for the key",
        "the blue circle holds the orange token up at a second gate to a room with a grey cylinder (the data)",
        "the gate opens and the blue circle walks through with the orange token, the grey cylinder is in reach"
      ],
      "change_panels": [
        "The grey key has turned into a round orange token held by the blue circle; the blue gate is unchanged.",
        "The blue circle has moved right to the blue gate and holds the round orange token against the gate's lock; a grey cylinder now appears behind the blue gate.",
        "The blue gate is now open, swung wide to both sides, and the blue circle has moved through it and stands next to the grey cylinder, holding the round orange token."
      ]
    },
    {
      "id": "queue",
      "title": "A queue with workers",
      "seed": 1006,
      "group": "worked",
      "panels": [
        "many small orange cubes (the jobs) arrive from the left and line up in a row on a grey conveyor belt",
        "the row of orange cubes moves along the grey conveyor belt toward two blue robot arms (the workers)",
        "each blue robot arm picks up one orange cube from the front of the row",
        "the blue robot arms place finished green cubes on a shelf at the right, the row of orange cubes is shorter"
      ],
      "change_panels": [
        "The row of orange cubes has moved to the right along the grey conveyor belt, and two blue robot arms now stand at the right end of the belt.",
        "Each blue robot arm now holds one orange cube lifted above the belt; the row of orange cubes on the belt is shorter.",
        "The blue robot arms are now empty and rest at the sides; a group of green cubes sits on a shelf at the right end of the belt and only a few orange cubes remain on the belt."
      ]
    },
    {
      "id": "ci",
      "title": "CI build pipeline",
      "seed": 1010,
      "group": "worked",
      "panels": [
        "a small orange cube (the commit) sits at the left start of a long grey pipe made of three sections",
        "the orange cube travels into the first section of the grey pipe, which glows blue, as it builds",
        "the orange cube passes into the second section of the pipe, which glows blue, as it is tested",
        "the orange cube exits the end of the pipe as a green cube on a shelf at the right, a green glow around it"
      ],
      "change_panels": [
        "The orange cube is now inside the first section of the grey pipe, and that first section glows bright blue.",
        "The orange cube is now inside the second section of the grey pipe; the first section is grey again and the second section glows bright blue.",
        "The orange cube has left the grey pipe and is now a green cube on a small shelf at the right end of the pipe; the whole pipe is grey and a green glow surrounds the green cube."
      ]
    },
    {
      "id": "merge",
      "title": "A diff and merge",
      "seed": 1012,
      "group": "worked",
      "panels": [
        "two stacks of coloured bars, one orange and one blue, stand side by side, each stack nearly the same as the other",
        "one bar in the orange stack and one bar in the blue stack are highlighted with a yellow glow, the rest are dimmed",
        "an arrow carries the highlighted blue bar over the orange stack to replace its highlighted bar",
        "a single combined stack of bars stands in the middle with a green glow, the two original stacks are faded behind it"
      ],
      "change_panels": [
        "One bar in the orange stack and one bar in the blue stack now have a yellow glow; all the other bars are unchanged.",
        "The glowing blue bar is now sitting in the orange stack in place of the glowing orange bar, which is gone; the blue stack has one empty gap.",
        "The two original stacks have faded to pale colours at the sides, and one new combined stack of bars stands in the middle with a green glow."
      ]
    }
  ]
}'''
import json
spec = json.loads(CONCEPTS_JSON)
print(len(spec['concepts']), 'concepts')

In [ ]:
# Download weights once (so the two workers do not race on the cache) and time it.
import os, time
from huggingface_hub import snapshot_download
t0 = time.time()
try:
    path = snapshot_download(spec["model"])
except Exception as e:
    print("DOWNLOAD FAILED:", repr(e))
    raise
DOWNLOAD_S = round(time.time() - t0, 1)
print("downloaded to", path, "in", DOWNLOAD_S, "s")
print(subprocess.run(f"du -sh {path}/*", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%writefile /kaggle/working/worker.py
"""One FLUX.2 klein 4B worker, pinned to one GPU by CUDA_VISIBLE_DEVICES.

Usage: python worker.py <spec.json> <out_dir> <result.json>

Stage 1 loads only the text encoder, encodes every prompt this worker needs, then frees it.
Stage 2 loads only the transformer + VAE. This keeps a T4 (15 GB) inside memory without CPU offload.
Dtype ladder if an image comes out black/NaN: fp16 -> fp16 + fp32 VAE -> bf16 + fp32 VAE.
"""
import gc
import json
import os
import shutil
import sys
import time

import numpy as np
import torch
from PIL import Image

spec = json.load(open(sys.argv[1]))
OUT = sys.argv[2]
RESULT = sys.argv[3]
M = spec["model"]
W, H, STEPS = spec["width"], spec["height"], spec["steps"]
jobs = spec["jobs"]
LADDER = [("fp16", torch.float16, False), ("fp16+vae32", torch.float16, True), ("bf16+vae32", torch.bfloat16, True)]

from diffusers import Flux2KleinPipeline  # noqa: E402

res = {"gpu": torch.cuda.get_device_name(0), "gpu_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
       "worker": spec["worker"], "load": {}, "images": [], "events": []}


def log(*a):
    print(f"[w{spec['worker']}]", *a, flush=True)


def save_result():
    json.dump(res, open(RESULT, "w"), indent=1)


def gb(x):
    return round(x / 1e9, 2)


def free():
    gc.collect()
    torch.cuda.empty_cache()


def bad(img):
    a = np.asarray(img).astype(np.float32)
    if not np.isfinite(a).all():
        return True
    return bool(a.mean() < 5 or a.std() < 2)


# ---- stage 1: text embeddings (computed once per unique prompt)
prompts = []
for j in jobs:
    if j["prompt"] not in prompts:
        prompts.append(j["prompt"])


def encode_all(dtype):
    t0 = time.perf_counter()
    pipe = Flux2KleinPipeline.from_pretrained(M, transformer=None, vae=None, dtype=dtype)
    pipe.text_encoder.to("cuda")
    torch.cuda.synchronize()
    res["load"]["text_encoder_load_s"] = round(time.perf_counter() - t0, 1)
    torch.cuda.reset_peak_memory_stats()
    t1 = time.perf_counter()
    embeds = {}
    with torch.no_grad():
        for p in prompts:
            e, _ = pipe.encode_prompt(prompt=p, device="cuda")
            embeds[p] = e.cpu()
    torch.cuda.synchronize()
    res["load"]["text_encode_s"] = round(time.perf_counter() - t1, 1)
    res["load"]["text_encode_n_prompts"] = len(prompts)
    res["load"]["text_stage_peak_gb"] = gb(torch.cuda.max_memory_allocated())
    res["load"]["text_dtype"] = str(dtype)
    del pipe
    free()
    return embeds


embeds = encode_all(torch.float16)
if any((not torch.isfinite(e).all()) for e in embeds.values()):
    res["events"].append("fp16 text embeddings contained NaN/inf; re-encoding text in bf16")
    log(res["events"][-1])
    embeds = encode_all(torch.bfloat16)
log("text stage done", res["load"])


# ---- stage 2: transformer + VAE
def load_pipe(level):
    name, dtype, vae32 = LADDER[level]
    t0 = time.perf_counter()
    pipe = Flux2KleinPipeline.from_pretrained(M, text_encoder=None, tokenizer=None, dtype=dtype)
    pipe.to("cuda")
    if vae32:
        pipe.vae.to(torch.float32)
    pipe.set_progress_bar_config(disable=True)
    torch.cuda.synchronize()
    res["load"][f"diffusion_load_s_{name}"] = round(time.perf_counter() - t0, 1)
    log("loaded", name, res["load"][f"diffusion_load_s_{name}"], "s; vram", gb(torch.cuda.memory_allocated()))
    return pipe


def generate(pipe, level, job, ref):
    name, dtype, vae32 = LADDER[level]
    g = torch.Generator(device="cuda").manual_seed(job["seed"])
    emb = embeds[job["prompt"]].to("cuda", dtype=dtype)
    kw = dict(prompt=None, prompt_embeds=emb, height=H, width=W, num_inference_steps=STEPS, guidance_scale=1.0, generator=g)
    if ref is not None:
        kw["image"] = ref
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        if vae32:
            lat = pipe(output_type="latent", **kw).images
            x = pipe.vae.decode(lat.to(torch.float32), return_dict=False)[0]
            img = pipe.image_processor.postprocess(x, output_type="pil")[0]
        else:
            img = pipe(**kw).images[0]
    torch.cuda.synchronize()
    return img, time.perf_counter() - t0


level = 0
pipe = load_pipe(level)
torch.cuda.reset_peak_memory_stats()
made = {}  # (concept, mode, panel) -> PIL
first = True
for job in jobs:
    key = (job["concept"], job["mode"], job["panel"])
    if job.get("copy_from"):
        src = tuple(job["copy_from"])
        made[key] = made[src]
        made[key].save(os.path.join(OUT, job["file"]))
        res["images"].append({"file": job["file"], "concept": job["concept"], "mode": job["mode"], "panel": job["panel"],
                              "copied_from": "_".join(map(str, src)), "seconds": 0.0, "counted": False})
        continue
    ref = made[tuple(job["ref"])] if job.get("ref") else None
    attempts = []
    while True:
        try:
            img, secs = generate(pipe, level, job, ref)
        except torch.cuda.OutOfMemoryError as e:
            res["events"].append(f"OOM on {job['file']}: {e}")
            log(res["events"][-1])
            raise
        isbad = bad(img)
        a = np.asarray(img).astype(np.float32)
        attempts.append({"dtype": LADDER[level][0], "seconds": round(secs, 3), "bad": isbad, "mean": round(float(a.mean()), 2), "std": round(float(a.std()), 2)})
        if not isbad or level == len(LADDER) - 1:
            break
        res["events"].append(f"{job['file']} was black/NaN in {LADDER[level][0]} (mean {a.mean():.2f}, std {a.std():.2f}); escalating")
        log(res["events"][-1])
        level += 1
        del pipe
        free()
        pipe = load_pipe(level)
    img.save(os.path.join(OUT, job["file"]))
    made[key] = img
    last = attempts[-1]
    res["images"].append({"file": job["file"], "concept": job["concept"], "mode": job["mode"], "panel": job["panel"],
                          "ref": bool(ref is not None), "dtype": last["dtype"], "seconds": last["seconds"], "bad": last["bad"],
                          "mean": last["mean"], "std": last["std"], "attempts": attempts, "first_image": first,
                          "counted": True, "peak_vram_gb": gb(torch.cuda.max_memory_allocated()),
                          "peak_reserved_gb": gb(torch.cuda.max_memory_reserved())})
    first = False
    log(f"{job['file']} {last['seconds']:.2f}s {last['dtype']} bad={last['bad']} mean={last['mean']}")
    save_result()

res["peak_vram_gb"] = gb(torch.cuda.max_memory_allocated())
res["peak_reserved_gb"] = gb(torch.cuda.max_memory_reserved())
res["final_dtype"] = LADDER[level][0]
res["done"] = True
save_result()
log("done")

In [ ]:
import json, os, signal, subprocess, sys, time
OUT = "/kaggle/working/out2"
os.makedirs(OUT, exist_ok=True)
STYLE, REFI, CHG = spec["style_prefix"], spec["reference_instruction"], spec["change_instruction"]
n_workers = max(1, N_GPU)
worker_jobs = [[] for _ in range(n_workers)]
for ci, c in enumerate(spec["concepts"]):
    js = worker_jobs[ci % n_workers]
    cid, seed = c["id"], c["seed"]
    js.append(dict(concept=cid, mode="p1", panel=1, seed=seed, prompt=f"{STYLE} {c['panels'][0]}", file=f"{cid}_p1_1.png"))
    for m in ("c", "d", "e"):
        js.append(dict(concept=cid, mode=m, panel=1, seed=seed, prompt=js[0]["prompt"], file=f"{cid}_{m}_1.png", copy_from=[cid, "p1", 1]))
    for p in range(2, 5):
        js.append(dict(concept=cid, mode="c", panel=p, seed=seed, prompt=f"{STYLE} {REFI} {c['panels'][p-1]}", file=f"{cid}_c_{p}.png", ref=[cid, "c", p-1]))
        dp = f"{STYLE} {CHG} {c['change_panels'][p-2]}"
        js.append(dict(concept=cid, mode="d", panel=p, seed=seed, prompt=dp, file=f"{cid}_d_{p}.png", ref=[cid, "d", p-1]))
        js.append(dict(concept=cid, mode="e", panel=p, seed=seed, prompt=dp, file=f"{cid}_e_{p}.png", ref=[cid, "p1", 1]))
print("jobs per worker:", [sum(1 for j in w if not j.get('copy_from')) for w in worker_jobs], "(generated images; copies of panel 1 excluded)")

procs = []
t_launch = time.time()
for w in range(n_workers):
    sp = {k: spec[k] for k in ("model", "width", "height", "steps")}
    sp.update(worker=w, jobs=worker_jobs[w])
    json.dump(sp, open(f"/kaggle/working/spec_w{w}.json", "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(w) if N_GPU else "")
    logf = open(f"/kaggle/working/worker_{w}.log", "w")
    procs.append(subprocess.Popen([sys.executable, "/kaggle/working/worker.py", f"/kaggle/working/spec_w{w}.json", OUT, f"/kaggle/working/result_w{w}.json"],
                                  stdout=logf, stderr=subprocess.STDOUT, env=env, start_new_session=True))
    print("launched worker", w, "pid", procs[-1].pid)

def kill_all():
    for p in procs:
        if p.poll() is None:
            try: os.killpg(p.pid, signal.SIGTERM)
            except ProcessLookupError: pass
    time.sleep(3)
    for p in procs:
        if p.poll() is None:
            try: os.killpg(p.pid, signal.SIGKILL)
            except ProcessLookupError: pass

seen = [0] * n_workers
try:
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        for w in range(n_workers):
            lines = open(f"/kaggle/working/worker_{w}.log").read().splitlines()
            for l in lines[seen[w]:]:
                if "Loading" not in l: print(l[:300])
            seen[w] = len(lines)
        if time.time() - t_launch > 3 * 3600:
            print("TIMEOUT"); break
finally:
    kill_all()
WALL_S = round(time.time() - t_launch, 1)
for w in range(n_workers):
    for l in open(f"/kaggle/working/worker_{w}.log").read().splitlines()[seen[w]:]:
        if "Loading" not in l: print(l[:300])
print("worker exit codes:", [p.returncode for p in procs], "wall", WALL_S, "s")

In [ ]:
import json, os, statistics
workers = [json.load(open(f"/kaggle/working/result_w{w}.json")) for w in range(n_workers) if os.path.exists(f"/kaggle/working/result_w{w}.json")]
imgs = [i for r in workers for i in r["images"] if i.get("counted")]
def mstat(mode):
    s = [i["seconds"] for i in imgs if i["mode"] == mode and not i.get("first_image")]
    return {"n": len([i for i in imgs if i["mode"] == mode]), "mean_s": round(statistics.mean(s), 2) if s else None}
per_gpu = []
for r in workers:
    gi = [i for i in r["images"] if i.get("counted")]
    st = [i["seconds"] for i in gi if not i.get("first_image")]
    per_gpu.append({"worker": r["worker"], "gpu": r["gpu"], "images": len(gi), "final_dtype": r.get("final_dtype"), "load": r["load"], "events": r["events"], "done": r.get("done", False),
                    "peak_vram_allocated_gb": r.get("peak_vram_gb"), "peak_vram_reserved_gb": r.get("peak_reserved_gb"),
                    "sec_per_image_steady": round(statistics.mean(st), 2) if st else None,
                    "images_per_hour_steady": round(3600 / statistics.mean(st), 1) if st else None})
results2 = {"gpus": GPU_INFO, "model": spec["model"], "resolution": "512x512", "steps": spec["steps"], "wall_generation_s": WALL_S,
            "images_generated": len(imgs), "by_mode": {m: mstat(m) for m in ("p1", "c", "d", "e")},
            "black_or_nan_first_try": sum(1 for i in imgs if i["attempts"][0]["bad"]), "black_or_nan_final": sum(1 for i in imgs if i["bad"]),
            "reference_strength_knob": "none: Flux2KleinPipeline.__call__ has no strength parameter and guidance_scale is ignored for the step-distilled klein",
            "per_gpu": per_gpu, "images": imgs}
json.dump(results2, open(f"{OUT}/results2.json", "w"), indent=1)
print(json.dumps({k: v for k, v in results2.items() if k not in ("per_gpu", "images")}, indent=1))
import shutil
for w in range(n_workers): shutil.copy(f"/kaggle/working/worker_{w}.log", f"{OUT}/worker_{w}.log")
shutil.make_archive("/kaggle/working/panels2", "zip", OUT)
print(len(os.listdir(OUT)), "files in out2")
print(subprocess.run("ps aux | grep -c '[w]orker.py'", shell=True, capture_output=True, text=True).stdout.strip(), "worker.py processes still running")